# Salt Lake City AQI Predictions and Forecasting

*Basic Info:*                                                                      

Title: Salt Lake City AQI Predictions and Forecasting.                                                 

Team Members:                                                         
+ Michael Northrup, u0992000,  michael.northrup@yahoo.com
+ Grant (Rob) Olsen, u0684253, rolsenrob@gmail.com
+ Dzevad Fitozovic, u0191522, dfitozovic@hotmail.com



## Background and Motivation.

We think that the quality of our air here in the valley is often a topic of discussion. However, the primary motivation came from one of our assignments. There we attempted to use the AQI to give us a moving average in order to be able to issue a warning to sensitive groups when an upwards trend was most likely occurring. We wanted to improve and expand on that, so in this project we want to see if we can predict the AQI for a few days in advance, creating a forecast for up to a week in advance. We also wish to see if we can make an AQI long term forecast model, so that one would be able to see how many days of poor air quality we can expect to see as our valley population rises. We do not have a background nor have we done any research in the area, but it seemed like an interesting problem. 

## Project Objectives.

Our primary objective is to devise a model to obtain an AQI forecast much like we have a weather forecast.  Additionally, we wanted to know how population growth in the valley will be affecting the number of days the AQI will reach over 101, however due to complexities with the forcast model we decided to drop this part of the project. Short term AQI data combined with weather forecasts should be useful in creating a short term forecast model. A short term model can be useful in warning sensitive groups of upcoming poor air quality days. And, if we can create a good predictive long term model, then it can be used as supporting evidence for a need to make a valley wide plan for future energy consumption that could limit the amount of pollutants like PM2.5 which accumulate due to our natural inversion conditions. 

## Deviations from Plans 

We originally planed on doing a 1-5 day AQI forecast model and a 1-5 year population growth model. However, due to the challenges with the 1-5 day model we decided to focus our energy there. Additionally, we intended to get data for weather and AQI going back to 1990. However, weather data going back that far was too unreliable, since there was a lot of missing data. We therefore fetched monthly reports from http://w2.weather.gov/climate/index.php?wfo=slc than formatted the data outside of python. Unfortunately, by this method we were only able to obtain 5 years of data going back to 11-01-2011. We believe that this should be enough for our model. 

## Data.

We will need to gather and clean a significant amount of data from several sources. For the long term
projection model the air quality data will be collected from https://www.epa.gov/outdoor-air-quality-data.
However, each year comes as a separate data-file, so it will need to be cleaned and combined.
To calculate projections for population growth we already obtained Salt Lake Valley census data from
http://www.census.gov/popest/data/index.html, and followed links to obtain historical data for years after 2010. Data formats from this source varied since the most recent years use newer data formats. Therefore, much of this data was formated and cleaned outside of the notebook. For our short term forecasting we will need an accurate 1-5 day weather forecast for the valley and previous days AQI, which we'll obtain by either web scraping http://www.weather.gov/ or more likely the API from http://openweathermap.org/. However, we may need further data sources as we continue to process the data from these sites. Some preliminary data analysis, formatting and cleanup was included in this
proposal.

In [1]:
# imports 
import pandas as pd
import requests
import json
import uuid
from IPython.display import display_javascript, display_html, display

import matplotlib.pyplot as plt 
%pylab inline
%matplotlib inline  
plt.rcParams['figure.figsize'] = (16,12)
plt.style.use('ggplot')


Populating the interactive namespace from numpy and matplotlib


In [2]:
# upload API data and store it in a dictionary 
response = requests.get("http://api.openweathermap.org/data/2.5/forecast?id=5780993&APPID=e0c55e00bf021f6142de334823046e9e&units=imperial")
weather = response.content.decode("utf-8")
weatherDict = json.loads(weather)


In [3]:
# print the raw dump file
print(json.dumps(weatherDict, indent=2))

{
  "message": 0.1885,
  "cod": "200",
  "cnt": 40,
  "city": {
    "population": 0,
    "sys": {
      "population": 0
    },
    "id": 5780993,
    "country": "US",
    "coord": {
      "lon": -111.891052,
      "lat": 40.76078
    },
    "name": "Salt Lake City"
  },
  "list": [
    {
      "snow": {},
      "dt": 1479492000,
      "clouds": {
        "all": 0
      },
      "weather": [
        {
          "description": "clear sky",
          "icon": "01d",
          "main": "Clear",
          "id": 800
        }
      ],
      "wind": {
        "speed": 6.78,
        "deg": 146
      },
      "main": {
        "temp_kf": 2.7,
        "pressure": 876.05,
        "humidity": 100,
        "grnd_level": 876.05,
        "sea_level": 1044.92,
        "temp_min": 35,
        "temp": 39.85,
        "temp_max": 39.85
      },
      "sys": {
        "pod": "d"
      },
      "dt_txt": "2016-11-18 18:00:00"
    },
    {
      "snow": {},
      "dt": 1479502800,
      "clouds": {
        "al

## Data Processing. 

Most of the cleanup for the population data and short term climate data was completed in OpenOffice out of convenience. Data clean up in the long term model air quality data set is expected to be comparable to the cleanup in Homework #3, although there is a lot more data which needed to be cleaned and combined. Also, we saw even more complexity in data cleanup of the population growth data and we expect substantial data cleanup in use of the API for the most recent weather forecasts. We expect that we should be able to give a fairly accurate short term AQI prediction, namely what type of air quality day one expects to have in the upcoming days, and long term projections based on population growth of at least 1-5 years in advance, assuming a reasonable amount of uncertainty and that current trends continue indefinitely.

In [4]:
# Method for rendering collapsible JSON from: http://stackoverflow.com/questions/18873066/pretty-json-formatting-in-ipython-notebook

class RenderJSON(object):
    def __init__(self, json_data):
        if isinstance(json_data, dict):
            self.json_str = json.dumps(json_data)
        else:
            self.json_str = json
        self.uuid = str(uuid.uuid4())

    def _ipython_display_(self):
        display_html('<div id="{}" style="height: 600px; width:100%;"></div>'.format(self.uuid),
        raw=True)
        
        display_javascript("""
        require(["https://rawgit.com/caldwell/renderjson/master/renderjson.js"], function() {
        document.getElementById('%s').appendChild(renderjson(%s))
        });
        """ % (self.uuid, self.json_str), raw=True)
        
RenderJSON(weatherDict)

In [5]:
#pdWeather = pd.read_json(weatherDict)
#pdWeather
#pd.DataFrame(weatherDict["data"], columns=[x["label"] for x in weatherDict["fields"]])

timeList = []
maxTempList = []
minTempList = []
pressureList = []
humidityList = []
tempList = []
wind_speedList = []
windDegList = []
precipitationList = []
rainList = []
rainListMM = []
snowList = []
snowListMM = []

for weatherEntry in weatherDict["list"]:
    timeList.append(weatherEntry["dt_txt"])
    mainWeather = weatherEntry["main"]
    maxTempList.append(mainWeather["temp_max"])
    minTempList.append(mainWeather["temp_min"])
    pressureList.append(mainWeather["pressure"])
    humidityList.append(mainWeather["humidity"])
    tempList.append(mainWeather["temp"])
    windWeather = weatherEntry["wind"]
    wind_speedList.append(windWeather["speed"])
    windDegList.append(windWeather["deg"])
    precip = 0
    if "rain" in weatherEntry:
        if "3h" in weatherEntry["rain"]:
            rainList.append(1)
            rainListMM.append(weatherEntry["rain"]["3h"])
            precip = precip + weatherEntry["rain"]["3h"]
    else:
        rainList.append(0)
        rainListMM.append(0)
    if "snow" in weatherEntry:
        if "3h" in weatherEntry["snow"]:
            snowList.append(1)
            snowListMM.append(weatherEntry["snow"]["3h"])
            precip = precip + weatherEntry["snow"]["3h"]
    else:
        snowList.append(0)
        snowListMM.append(0)
    precipitationList.append(precip)    
    

In [6]:
precipitationList

[0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0.001,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0.52,
 0.29,
 0.06,
 0.02,
 0.01,
 0.03,
 0.03,
 0.03,
 0.05,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [8]:
# setup of dataframe for 5 day forecast
data = [('DateTime', timeList),
         ('MaxTemp', maxTempList),
         ('MinTemp', minTempList),
         ('Pressure', pressureList),
         ('Humidity', humidityList),
         ('Temperature', tempList),
         ('WindSpeed', wind_speedList), 
         ('WindDeg', windDegList),
         ('Precipitation (mm)', precipitationList)
         ]

weatherForecast = pd.DataFrame.from_items(data)
weatherForecast["DateTime"] = weatherForecast["DateTime"].apply(lambda x: str(x)[:10])

finalForecast = weatherForecast.groupby("DateTime").mean()

finalForecast["MinTemp"] = weatherForecast.groupby("DateTime").min()["MinTemp"]
finalForecast["MaxTemp"] = weatherForecast.groupby("DateTime").max()["MaxTemp"]
finalForecast["MaxPressure"] = weatherForecast.groupby("DateTime").max()["Pressure"]
finalForecast["MinPressure"] = weatherForecast.groupby("DateTime").min()["Pressure"]
finalForecast["MaxWindSpeed"] = weatherForecast.groupby("DateTime").max()["WindSpeed"]
finalForecast["MinWindSpeed"] = weatherForecast.groupby("DateTime").min()["WindSpeed"]

print(finalForecast.head())

            MaxTemp  MinTemp   Pressure  Humidity  Temperature  WindSpeed  \
DateTime                                                                    
2016-11-18    43.09    35.00  875.13000   100.000     41.47000    5.68500   
2016-11-19    49.45    30.17  871.16875    98.875     36.24500    6.34500   
2016-11-20    55.18    41.14  867.69125    90.750     46.13125    8.30750   
2016-11-21    51.39    43.89  865.04250    92.875     46.87250   11.29375   
2016-11-22    45.71    36.56  869.13375   100.000     40.96000    4.13625   

               WindDeg  Precipitation (mm)  MaxPressure  MinPressure  \
DateTime                                                               
2016-11-18  144.002500            0.000000       876.05       874.21   
2016-11-19  147.003000            0.000000       872.31       869.00   
2016-11-20  171.566250            0.000125       868.16       866.19   
2016-11-21  164.188750            0.101250       865.85       864.50   
2016-11-22  249.314125      

In [ ]:
# Load AQI data from files and group by mean for each date so that we only have 1 value for each date.
aqi_data_2011 = pd.read_csv("2011.csv",parse_dates=True)
df2011 = pd.DataFrame(aqi_data_2011).groupby("Date").mean()

aqi_data_2012 = pd.read_csv("2012.csv",parse_dates=True)
df2012 = pd.DataFrame(aqi_data_2012).groupby("Date").mean()

aqi_data_2013 = pd.read_csv("2013.csv",parse_dates=True)
df2013 = pd.DataFrame(aqi_data_2013).groupby("Date").mean()

aqi_data_2014 = pd.read_csv("2014.csv",parse_dates=True)
df2014 = pd.DataFrame(aqi_data_2014).groupby("Date").mean()

aqi_data_2015 = pd.read_csv("2015.csv",parse_dates=True)
df2015 = pd.DataFrame(aqi_data_2015).groupby("Date").mean()

aqi_data_2016 = pd.read_csv("2016.csv",parse_dates=True)
df2016 = pd.DataFrame(aqi_data_2016).groupby("Date").mean()

aqi_data_present = pd.read_csv("AQIOctNov.csv",parse_dates=True)
dfpresent = pd.DataFrame(aqi_data_present).groupby("Date").mean()

# Combine data into a single dataframe
all_aqi_data = pd.DataFrame()

all_aqi_data = df2011.append(df2012)
all_aqi_data = all_aqi_data.append(df2013)
all_aqi_data = all_aqi_data.append(df2014)
all_aqi_data = all_aqi_data.append(df2015)
all_aqi_data = all_aqi_data.append(df2016)
all_aqi_data = all_aqi_data.append(dfpresent)

# Forward fill empty values
# idx = pd.date_range('11-01-2011', '11-16-2016')
# all_aqi_data.index = pd.DatetimeIndex(all_aqi_data.index)
# all_aqi_data = all_aqi_data.reindex(idx,method='ffill') 
all_aqi_data = all_aqi_data.fillna(method = 'ffill')

# Create moving average columns.
all_aqi_data["AQI 14d avg"] = np.round(all_aqi_data["DAILY_AQI_VALUE"].rolling(window = 14, center = False).mean(), 2)
all_aqi_data["AQI 7d avg"] = np.round(all_aqi_data["DAILY_AQI_VALUE"].rolling(window = 7, center = False).mean(), 2)
all_aqi_data["AQI 3d avg"] = np.round(all_aqi_data["DAILY_AQI_VALUE"].rolling(window = 3, center = False).mean(), 2)

# Remove unused columns
del all_aqi_data["POC"]
del all_aqi_data["AQS_SITE_ID"]
del all_aqi_data["Daily Mean PM2.5 Concentration"]
del all_aqi_data["DAILY_OBS_COUNT"]
del all_aqi_data["PERCENT_COMPLETE"]
del all_aqi_data["AQS_PARAMETER_CODE"]
del all_aqi_data["CBSA_CODE"]
del all_aqi_data["STATE_CODE"]
del all_aqi_data["COUNTY_CODE"]
del all_aqi_data["SITE_LATITUDE"]
del all_aqi_data["SITE_LONGITUDE"]

# remove data prior to 11/01/2011 to match up with aquired weather data
all_aqi_data = all_aqi_data ["11/01/2011":]
all_aqi_data.head()

In [ ]:
#visualization
all_aqi_data["DAILY_AQI_VALUE"].plot(grid = True, color = 'b');
all_aqi_data["AQI 14d avg"].plot(grid = True, color = 'g');
all_aqi_data["AQI 7d avg"].plot(grid = True, color = 'r');
all_aqi_data["AQI 3d avg"].plot(grid = True, color = 'y');

In [ ]:
# load climate data 
weather_data_slc = pd.read_csv("recent_climate_data.csv")
# change Percipitation and Snow units from inches to mm to fit in with 5 day forecast data 
weather_data_slc['Percipitation'] = weather_data_slc['Percipitation']*25.4
weather_data_slc['Snow'] = weather_data_slc['Snow']*25.4
# reindex by day of month. 
weather_data_slc = weather_data_slc.set_index(["Date"])
weather_data_slc.describe()

## Exploratory Analysis.

We will visualize how precipitation, temperature and wind speed effect the AQI by plotting. 

In [ ]:
# add AQI data to weather data
frames = [weather_data_slc, all_aqi_data]
all_historic_data = pd.concat(frames, axis = 1)
# create column of dates
all_historic_data.describe()
# plot of AQI and Temperature
all_historic_data['DAILY_AQI_VALUE'].plot(grid = True, color = 'b'); 
all_historic_data['Percipitation'].plot(grid = True, color = 'r');


In [ ]:
# plot of AQI and Avg. Wind speed
all_historic_data['DAILY_AQI_VALUE'].plot(grid = True, color = 'b'); 
all_historic_data['Avgwindspeed(mph)'].plot(grid = True, color = 'y');

In [ ]:
# plot of AQI and Max Temperature
all_historic_data['DAILY_AQI_VALUE'].plot(grid = True, color = 'b');
all_historic_data['MAXtemp(F)'].plot(grid = True, color = 'g'); 

## Analysis Methodology.

For our short term model, we will combine the weather predictions and past month's data to create a forecast model for up to 5 days in advance. The historical data will be used to train and test the model. Additionally, the predictions will be recursive, since the AQI for the last predicted day will also use the predicted data from previous 4 days, and so on. We will use cross validation on several different models to see which will work best. Due to the fact that our weather data comes from different sources, only select few features were able to be used, therefore doing further dimensionality reduction will not be needed.

In [ ]:
# get needed imports 
import numpy as np
from sklearn import tree, svm, metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cross_validation import train_test_split, cross_val_predict, cross_val_score, KFold

In [ ]:
# create a boolian column for AQI greater than or less than 101
def AQI_to_numeric(a):
    if a > 101:
        return 1
    else:
        return 0
all_historic_data["Boolean AQI"]=all_historic_data["DAILY_AQI_VALUE"].apply(AQI_to_numeric)
all_historic_data = all_historic_data.fillna(method = 'ffill')
# convert to data to a matrix
y = all_historic_data["Boolean AQI"].as_matrix()
X = all_historic_data.drop(["DAILY_AQI_VALUE","Boolean AQI"], axis=1).as_matrix()


In [ ]:
# split data 
features = list(all_historic_data)
def splitData(features):
    predctors = X
    labels = y

    # Split into training and test sets 
    XTrain, XTest, yTrain, yTest = train_test_split(predctors, labels, random_state=1, test_size=0.3)
    return XTrain, XTest, yTrain, yTest

XTrain, XTest, yTrain, yTest = splitData(features)


In [ ]:
# preliminary model to test historic data. 
# cross validaton for decission tree model
depth_list = [1,2,3,4,5,6,7,8,9]
sample_split_list = [5,10,15,20,25,30,50]  

for item1 in sample_split_list:
    print("__________________________________________________________________________________________ ")
    print ("Min. Sample Split: ", item1)
    for item2 in depth_list: 
        decisionTree = tree.DecisionTreeClassifier(max_depth=item2, min_samples_split=item1)
        decisionTree = decisionTree.fit(XTrain, yTrain)
        print("Max Depth: ", item2)
        y_pred_train = decisionTree.predict(XTrain)
        print('Accuracy on training data= ', metrics.accuracy_score(y_true = yTrain, y_pred = y_pred_train))
        y_pred = decisionTree.predict(XTest)
        print('Accuracy on test data= ', metrics.accuracy_score(y_true = yTest, y_pred = y_pred))


## Project Schedule. 

We plan on mostly working together to achieve each project deadline. However, some work will need to be divided out if we are unable to reach each week's objective during team meetings. 

By Friday 11-04-16
+ Obtain and format long term climate, population and AQI data. (Rob, Michael and Dzevad)
+ Obtain and format short term climate and AQI data. (Rob, Michael and Dzevad)
+ Obtain current weather forecast using an API. (Rob, Michael and Dzevad)

By Friday 11-11-16 
+ Create a long term forecasting model and test it on year 2015. (Rob, Michael and Dzevad) 
+ Create a short term forecasting model and check if it is able to predict a few days ahead. (Rob, Michael and Dzevad) 

By Friday 11-18-16
+ Tested the short term and long term model. (Rob, Michael and Dzevad)
+ Created plots to help visualize the data. (Rob, Michael and Dzevad)
+ Turn in project milestone. (Rob, Michael and Dzevad)

By Wednesday 11-23-16 
+ Refine the short term and long term model. (Rob, Michael and Dzevad)
+ Begin creating Project Screen-Cast. (Rob, Michael and Dzevad)
+ Begin creating presentation. (Rob, Michael and Dzevad)

By Friday 12-02-16
+ Completed AQI perdition and forecasting project notebook. (Rob, Michael and Dzevad)
+ Completed Project Screen-Cast and presentation. (Rob, Michael and Dzevad)

